In [18]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [19]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()

X = data.data
y = data.target

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (20640, 8)
Target shape: (20640,)


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (16512, 8)
Testing data: (4128, 8)


In [21]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(
    y_train.reshape(-1, 1)
).ravel()

In [22]:
models = {
    "Linear Regression": LinearRegression(),
    "Polynomial Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "SVR": SVR(kernel='rbf')
}

In [23]:
poly = PolynomialFeatures(degree=2)

X_poly_train = poly.fit_transform(X_train)
X_poly_test = poly.transform(X_test)

print("Original features:", X_train.shape[1])
print("Polynomial features:", X_poly_train.shape[1])

Original features: 8
Polynomial features: 45


In [24]:
results = {}

for name, model in models.items():

    if name == "Polynomial Regression":

        model.fit(X_poly_train, y_train)
        y_pred = model.predict(X_poly_test)

    elif name == "SVR":

        model.fit(X_train_scaled, y_train_scaled)

        y_pred_scaled = model.predict(X_test_scaled)

        y_pred = scaler_y.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).ravel()

    else:

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    results[name] = {
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "R2 Score": r2_score(y_test, y_pred)
    }

In [25]:
results_df = pd.DataFrame(results).T

print(results_df)

                            MAE       MSE  R2 Score
Linear Regression      0.533200  0.555892  0.575788
Polynomial Regression  0.467001  0.464302  0.645682
Decision Tree          0.454679  0.495235  0.622076
Random Forest          0.327543  0.255368  0.805123
SVR                    0.397283  0.354151  0.729740


In [26]:
print("Regression Model Comparison")
print("=" * 60)

print(results_df)

print("\nBest R2 Score:")
print(results_df["R2 Score"].idxmax())

print("\nLowest MAE:")
print(results_df["MAE"].idxmin())

print("\nLowest MSE:")
print(results_df["MSE"].idxmin())

Regression Model Comparison
                            MAE       MSE  R2 Score
Linear Regression      0.533200  0.555892  0.575788
Polynomial Regression  0.467001  0.464302  0.645682
Decision Tree          0.454679  0.495235  0.622076
Random Forest          0.327543  0.255368  0.805123
SVR                    0.397283  0.354151  0.729740

Best R2 Score:
Random Forest

Lowest MAE:
Random Forest

Lowest MSE:
Random Forest
